# BERT Score Calculation for MXene Literature Mining

This notebook calculates the BERT score between predicted data and gold standard data.

In [ ]:
# Install required packages
!pip install bert-score torch

In [ ]:
# Upload the JSON files
from google.colab import files
print("Upload your transformed_data.json file:")
uploaded_pred = files.upload()
pred_filename = next(iter(uploaded_pred.keys()))

print("\nUpload your gold.json file:")
uploaded_gold = files.upload()
gold_filename = next(iter(uploaded_gold.keys()))

In [ ]:
import json
import torch
from bert_score import score

# Load the files
with open(pred_filename, 'r') as f:
    pred_data = json.load(f)

with open(gold_filename, 'r') as f:
    gold_data = json.load(f)

# Print basic stats
print(f"Predicted data: {len(pred_data)} entries")
print(f"Gold data: {len(gold_data)} entries")

In [ ]:
# Convert to strings for BERT score calculation
pred_texts = [json.dumps(entry) for entry in pred_data]
gold_texts = [json.dumps(entry) for entry in gold_data]

# Calculate BERT scores
P, R, F1 = score(pred_texts, gold_texts, lang="en", verbose=True)

# Print results
print(f"\nBERT Score Results:")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall: {R.mean().item():.4f}")
print(f"F1: {F1.mean().item():.4f}")

In [ ]:
# Calculate per-entry scores
print("\nPer-entry scores:")
for i, (p, r, f1) in enumerate(zip(P, R, F1)):
    print(f"Entry {i+1}: P={p.item():.4f}, R={r.item():.4f}, F1={f1.item():.4f}")

In [ ]:
# Install and import rouge package
!pip install rouge

from rouge import Rouge

print("\n\n# ROUGE Score Calculation")
print("------------------------")

# Initialize Rouge
rouge = Rouge()

# Calculate ROUGE scores
rouge_scores = rouge.get_scores(pred_texts, gold_texts, avg=True)

# Print results
print(f"\nROUGE Score Results:")
print(f"ROUGE-1 F1: {rouge_scores['rouge-1']['f']:.4f}")
print(f"ROUGE-2 F1: {rouge_scores['rouge-2']['f']:.4f}")
print(f"ROUGE-L F1: {rouge_scores['rouge-l']['f']:.4f}")

# Calculate per-entry ROUGE scores
print("\nPer-entry ROUGE scores:")
individual_scores = []
for i, (pred, gold) in enumerate(zip(pred_texts, gold_texts)):
    try:
        score = rouge.get_scores(pred, gold)[0]
        individual_scores.append(score)
        print(f"Entry {i+1}: ROUGE-1 F1={score['rouge-1']['f']:.4f}, ROUGE-2 F1={score['rouge-2']['f']:.4f}, ROUGE-L F1={score['rouge-l']['f']:.4f}")
    except Exception as e:
        print(f"Entry {i+1}: Error calculating ROUGE score - {str(e)}")

In [ ]:
# Visualize the comparison between BERT and ROUGE scores
import matplotlib.pyplot as plt
import numpy as np

# Extract BERT F1 scores
bert_f1_scores = [f1.item() for f1 in F1]

# Extract ROUGE-1 F1 scores for valid entries
rouge_f1_scores = [score['rouge-1']['f'] for score in individual_scores]

# Create indices for comparison (only where both scores exist)
valid_indices = min(len(bert_f1_scores), len(rouge_f1_scores))
indices = np.arange(valid_indices)

# Create the plot
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.bar(indices, bert_f1_scores[:valid_indices], alpha=0.7, label='BERT F1')
plt.bar(indices, rouge_f1_scores[:valid_indices], alpha=0.5, label='ROUGE-1 F1')
plt.xlabel('Entry Index')
plt.ylabel('Score')
plt.title('BERT vs ROUGE F1 Scores per Entry')
plt.legend()
plt.grid(True, alpha=0.3)

# Create a scatter plot to see correlation
plt.subplot(1, 2, 2)
plt.scatter(bert_f1_scores[:valid_indices], rouge_f1_scores[:valid_indices], alpha=0.7)
plt.xlabel('BERT F1 Score')
plt.ylabel('ROUGE-1 F1 Score')
plt.title('Correlation between BERT and ROUGE Scores')
plt.grid(True, alpha=0.3)

# Add correlation coefficient
if valid_indices > 1:  # Need at least 2 points for correlation
    corr = np.corrcoef(bert_f1_scores[:valid_indices], rouge_f1_scores[:valid_indices])[0,1]
    plt.annotate(f'Correlation: {corr:.2f}', xy=(0.05, 0.95), xycoords='axes fraction')

plt.tight_layout()
plt.show()

In [ ]:
# Create a summary table with both BERT and ROUGE metrics
import pandas as pd

# Create a DataFrame for summary metrics
summary_df = pd.DataFrame({
    'Metric': ['BERT Precision', 'BERT Recall', 'BERT F1', 
               'ROUGE-1 Precision', 'ROUGE-1 Recall', 'ROUGE-1 F1',
               'ROUGE-2 Precision', 'ROUGE-2 Recall', 'ROUGE-2 F1',
               'ROUGE-L Precision', 'ROUGE-L Recall', 'ROUGE-L F1'],
    'Value': [P.mean().item(), R.mean().item(), F1.mean().item(),
              rouge_scores['rouge-1']['p'], rouge_scores['rouge-1']['r'], rouge_scores['rouge-1']['f'],
              rouge_scores['rouge-2']['p'], rouge_scores['rouge-2']['r'], rouge_scores['rouge-2']['f'],
              rouge_scores['rouge-l']['p'], rouge_scores['rouge-l']['r'], rouge_scores['rouge-l']['f']]
})

# Display the summary table
print("\nEvaluation Metrics Summary:")
print("---------------------------")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Create a DataFrame for per-entry scores
if valid_indices > 0:
    entry_scores = []
    for i in range(valid_indices):
        entry_scores.append({
            'Entry': i+1,
            'BERT F1': bert_f1_scores[i],
            'ROUGE-1 F1': rouge_f1_scores[i],
            'ROUGE-2 F1': individual_scores[i]['rouge-2']['f'],
            'ROUGE-L F1': individual_scores[i]['rouge-l']['f']
        })
    
    entries_df = pd.DataFrame(entry_scores)
    
    # Display top 5 and bottom 5 entries by BERT F1 score
    print("\nTop 5 entries by BERT F1 score:")
    print(entries_df.sort_values('BERT F1', ascending=False).head(5).to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else x))
    
    print("\nBottom 5 entries by BERT F1 score:")
    print(entries_df.sort_values('BERT F1').head(5).to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else x))
    
    # Save the results to CSV
    entries_df.to_csv('evaluation_metrics.csv', index=False)
    print("\nDetailed evaluation metrics saved to 'evaluation_metrics.csv'")